# HVAC 2.0 — Stage 0: Pre-training and Source-Domain Baseline

Corresponds to `proposal/proposal.md` v6 §5 / §7, Stage 0. This notebook **covers Stage 0 only** and runs end to end with no manual intervention.

Produces **5 pre-trained models** + each model's **AUC / best F1** on its own 480-clip source-domain evaluation set (the source-domain baseline,
i.e. the reference that later stages subtract from when measuring "forgetting").

| Model | Training data | Train pool |
| --- | --- | --- |
| `id_00` / `id_02` / `id_04` / `id_06` | All normal clips of that machine ID minus its evaluation set | 851 / 856 / 873 / 855 |
| `merged` | 215 clips from each of the four IDs | 860 |

**How to run**: `Runtime → Run all`. Cells 1→9 execute in order; no parameters need to be edited by hand.

**Runtime**: `Edit → Notebook settings → T4 GPU` recommended. CPU also works, but training is 5–10× slower.

**Embedded config**: this notebook embeds a verbatim copy of `code/configs/config.yaml`,
sha256 = `8430e581ae6b4e14...`. The repository's `code/configs/config.yaml` remains the single source of truth;
if it is modified, this notebook must be regenerated.

> Note: this sha256 is the hash of the original config file. Translating its comments to English changes the hash,
> so a hash-mismatch warning against the archived results is expected and harmless.

## 1 · Environment and dependencies

Checks that torch / librosa are available and reports the compute device. If it shows `CPU`, training still works, just more slowly.

In [ ]:
import sys, os, time, json, glob, hashlib, zipfile, shutil, subprocess

def _ensure(pkg, imp=None):
    try:
        __import__(imp or pkg)
    except ImportError:
        print(f"Installing {pkg} ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

_ensure("pyyaml", "yaml")
_ensure("librosa")
_ensure("tqdm")

import numpy as np
import yaml
import librosa
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn import metrics
from sklearn.metrics import precision_recall_fscore_support, accuracy_score
from tqdm.auto import tqdm

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Python  :", sys.version.split()[0])
print("numpy   :", np.__version__)
print("librosa :", librosa.__version__)
print("torch   :", torch.__version__)
print("Device  :", DEVICE, "|", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "no GPU (training will be 5-10x slower)")

## 2 · Configuration (`code/configs/config.yaml` embedded verbatim)

Written to `/content/config.yaml` and loaded, then the exact same self-check as the repository's `code/configs/config.py` is run.
The self-check covers train-pool arithmetic, equal per-ID contribution to the merged model, matching source/target class ratios, feature dimension, LORO per-fold composition,
validity of the layer names used by the layer schemes, and the 272/992/1032 counts. Any mismatch **raises immediately**,
so hundreds of training runs never proceed with a wrong config.

In [ ]:
_CONFIG_YAML_RAW = r"""
# ============================================================================
# HVAC 2.0 — experiment configuration (single source of truth)
# Matches proposal/proposal.md v6. On any conflict, the proposal wins and this file is updated.
# This file holds only "constants and rules"; no training logic.
# ============================================================================

project: HVAC_2.0
proposal_version: v6
seed: 42                      # the one random seed for the whole project (proposal §4.1)

# ---------------------------------------------------------------------------
# Paths. env selects which block is used; keep colab when running on Colab, set local for local debugging.
# ---------------------------------------------------------------------------
paths:
  env: colab                  # colab | local

  colab:
    drive_root: /content/drive/MyDrive
    # Source and target domains are both zips on Drive. Always extract to the Colab local disk /content/data,
    # never back onto Drive: Drive small-file IO is extremely slow, and it would push thousands of wavs to the cloud.
    # The local disk is ephemeral and must be re-extracted after a runtime restart -- extraction should be idempotent (skip if the directory exists).
    extract_root: /content/data

    # Source domain: MIMII 0dB fan
    mimii_zip: /content/drive/MyDrive/HVAC_fan/0_dB_fan.zip
    mimii_extract_to: /content/data/mimii
    mimii_root: /content/data/mimii/fan          # contains id_00/id_02/id_04/id_06

    # Target domain: 180 self-recorded fan clips
    target_zip: /content/drive/MyDrive/HVAC_fan/raw_audio.zip
    target_extract_to: /content/data/target
    target_root: /content/data/target/raw_audio  # contains {condition}/{voltage}/{noise}/
    recording_log: null                          # inventory check only; metadata comes from filenames

    # Outputs: written back to Drive so they survive runtime restarts
    outputs_root: /content/drive/MyDrive/HVAC_fan/HVAC2_Results

  local:
    drive_root: null
    extract_root: null
    # Already extracted locally; no zips needed
    mimii_zip: null
    mimii_extract_to: null
    mimii_root: data/mimii/fan
    target_zip: null
    target_extract_to: null
    target_root: data/raw_audio
    recording_log: data/metadata/recording_log.csv
    outputs_root: outputs

outputs:
  # Stages 2+3 total ~1032 train/eval runs. Weights are discarded after use; only CSVs are saved (proposal §7, stage 2)
  persist_finetuned_weights: false
  pretrained_models_dir: models        # relative to outputs_root; these 5 weights must be kept
  results_dir: logs
  figures_dir: figures

# ---------------------------------------------------------------------------
# Source domain: MIMII fan 0dB (proposal §4.1)
# ---------------------------------------------------------------------------
source_domain:
  name: MIMII_fan_0dB
  machine_ids: [id_00, id_02, id_04, id_06]

  # Official counts, used for a startup self-check: scanned file counts must match exactly, else fail immediately
  official_counts:
    id_00: {normal: 1011, abnormal: 407}
    id_02: {normal: 1016, abnormal: 359}
    id_04: {normal: 1033, abnormal: 348}
    id_06: {normal: 1015, abnormal: 361}

  # Evaluation set: class ratio 1:2, aligned with the target domain's 60:120 (proposal §3)
  eval_set:
    n_normal: 160
    n_abnormal: 320

  # Training pool = all normal clips of that machine ID − the 160 used by the evaluation set
  train_pool_size:
    id_00: 851
    id_02: 856
    id_04: 873
    id_06: 855

  # Merged model: equal contribution from each machine ID
  merged_model:
    # Evaluation set must be a subset of each ID's fixed evaluation set; no independent re-draw (proposal §4.1)
    eval_per_id: {normal: 40, abnormal: 80}
    # Training set: 215 clips drawn from each ID's "training pool", so the total matches the single-ID models
    train_per_id: 215
    train_total: 860

# ---------------------------------------------------------------------------
# Target domain: 180 self-recorded fan clips (proposal §4.2)
# ---------------------------------------------------------------------------
target_domain:
  name: self_recorded_fan
  device: Razer Seiren V3 Mini

  conditions: [normal, blocked, imbalance]
  normal_conditions: [normal]                 # 60 clips, the only part that may enter training
  abnormal_conditions: [blocked, imbalance]   # 120 clips, evaluation only
  voltages: ["4V", "8V", "12V"]
  noise_levels: [quiet, noise]
  n_runs: 10

  n_clips_total: 180
  n_normal_total: 60
  n_abnormal_total: 120

  # Filenames are the single source of truth for metadata; recording_log.csv is only an inventory check
  # (the csv has 206 rows with duplicates and misaligned columns; after dedup, the 180 filenames form a complete 18×10 grid)
  filename_pattern: "{condition}_{voltage}_{noise}_run{run:02d}.wav"
  relpath_pattern: "{condition}/{voltage}/{noise}/{filename}"

# ---------------------------------------------------------------------------
# Feature extraction (proposal §5, identical to baseline_runner.ipynb)
# ---------------------------------------------------------------------------
features:
  sr: 16000
  duration_sec: 10
  n_mels: 64
  frames: 5           # concatenate 5 adjacent frames
  n_fft: 1024
  hop_length: 512
  power: 2.0
  input_dim: 320      # = n_mels * frames
  # Each 10 s clip yields ~309 vectors of dim 320 (313 frames − 5 + 1)

# ---------------------------------------------------------------------------
# Model architecture (proposal §5)
# ---------------------------------------------------------------------------
model:
  arch: MIMII_Baseline_AE
  encoder_dims: [320, 64, 64, 8]
  decoder_dims: [8, 64, 64, 320]
  bottleneck_dim: 8
  activation: relu
  output_activation: none      # official baseline has no activation on the output layer
  # Layer indices inside the torch module, used by the layer schemes to freeze/unfreeze by name
  layer_index:
    encoder_linears: ["encoder.0", "encoder.2", "encoder.4"]   # 320→64, 64→64, 64→8
    decoder_linears: ["decoder.0", "decoder.2", "decoder.4"]   # 8→64, 64→64, 64→320

# ---------------------------------------------------------------------------
# Pretraining (stage 0, proposal §5 / §7 stage 0)
# ---------------------------------------------------------------------------
pretrain:
  optimizer: adam
  lr: 0.001            # torch Adam default, same as baseline_runner.ipynb
  loss: mse
  batch_size: 512
  epochs: 50
  validation_split: 0.1
  shuffle: true
  # Right after training, evaluate once on the model's own 480-clip source eval set and store as the "source baseline" (subtrahend for forgetting)
  record_source_baseline: true

# ---------------------------------------------------------------------------
# Fine-tuning (stage 2) — all four layer schemes share one hyperparameter set, not tuned per scheme (proposal §7, stage 2)
# ---------------------------------------------------------------------------
finetune:
  optimizer: adam
  loss: mse
  lr: 0.0001           # 1/10 of pretraining lr; at the 40-clip full level the movement budget is still < 2.5% of pretraining
  epochs: 30           # derived from a step budget: ~750 updates at the 5-clip level, ~5820 at the 40-clip level
  batch_size: 64       # 512 would give only 4 batches per epoch at the 5-clip level, too few for Adam momentum to warm up
  source_replay: false          # no source-domain replay (proposal §7, stage 2)
  early_stopping: false         # no early stopping; rationale in proposal §7, stage 2
  restart_from_pretrained_each_fold: true   # every LORO fold must restart from the pretrained weights

# ---------------------------------------------------------------------------
# LORO validation protocol (proposal §6)
# ---------------------------------------------------------------------------
loro:
  applies_to: large_shift_only     # no LORO for small shift (proposal §6.4)
  n_folds: 10                      # fold k holds out the whole run == k+1
  fold_key: run                    # determined by runNN in the filename
  clips_per_fold: 18               # one clip from each of the 18 conditions
  normals_per_fold: 6
  abnormals_per_fold: 12
  max_train_normals: 54            # 60 − 6, which is why the data levels stop at 40
  report: [pooled, per_fold]       # compute both views (proposal §6.2)
  per_fold_stats: [mean, std]
  pooled_normalization: none       # no per-fold normalization; listed as a limitation

# ---------------------------------------------------------------------------
# Experiment grid (proposal §7, stage 2)
# ---------------------------------------------------------------------------
grid:
  data_levels: [5, 10, 20, 40]

  # Nested sampling: 5 ⊂ 10 ⊂ 20 ⊂ 40 (proposal §4.3)
  nested_sampling:
    enabled: true
    rule: "Candidates are put in a fixed order by a fixed rule; level n takes the first n"
    # Ordering = sort by filename for a deterministic initial order, then shuffle with a numpy RNG
    canonical_order: filename_sort_then_shuffle
    rng: numpy_default_rng
    # Small shift: order each target ID's training pool once (seed scope includes only the ID)
    small_shift_seed_scope: [seed, target_id]
    # Large shift: available clips differ per fold, so each fold is ordered separately (seed scope includes the fold)
    large_shift_seed_scope: [seed, fold]

  layer_schemes:
    full:
      desc: Full fine-tuning
      trainable: ["encoder.0", "encoder.2", "encoder.4", "decoder.0", "decoder.2", "decoder.4"]
    decoder_only:
      desc: Decoder only
      trainable: ["decoder.0", "decoder.2", "decoder.4"]
    encoder_only:
      desc: Encoder only
      trainable: ["encoder.0", "encoder.2", "encoder.4"]
    bottleneck_only:
      desc: Bottleneck-adjacent layers only (the two layers into and out of the bottleneck, 64→8 and 8→64)
      trainable: ["encoder.4", "decoder.0"]

# ---------------------------------------------------------------------------
# Experiment enumeration per stage (counts are self-checked at startup by config.py; mismatch raises)
# ---------------------------------------------------------------------------
stages:
  stage0_pretrain:
    n_models: 5                  # 4 single-ID + 1 merged
  stage1_frozen_eval:
    small_shift_pairs: 12        # 4 starts × 3 unseen IDs each
    large_shift_models: 5        # all 5 models evaluated on large shift (incl. merged)
    large_shift_folds: 10
    shift_ratio:
      aggregation: median
      denominator: source_eval_normals          # the 160 normals of that ID's eval set
      numerator_small_shift: target_eval_normals # the 160 normals of the target ID's eval set
      numerator_large_shift: all_60_normals      # frozen eval has no leakage, so use all 60 clips
  stage2_finetune:
    small_shift:
      starts: 4
      targets_per_start: 3
      folds: 1
      n_configs: 192
      n_runs: 192
    large_shift:
      starts: 5                  # 4 single-ID + 1 merged
      targets_per_start: 1
      folds: 10
      n_configs: 80
      n_runs: 800
    n_configs_total: 272
    n_runs_total: 992
  stage3_no_pretrain:
    init: random
    layer_scheme: full           # random init has no "learned layers", so no layer-scheme variable
    n_configs: 4
    folds: 10
    n_runs: 40
    source_metrics: null         # no source domain; reported as N/A

total_runs: 1032                 # 992 + 40

# ---------------------------------------------------------------------------
# Evaluation metrics (proposal §3)
# ---------------------------------------------------------------------------
metrics:
  primary: auc                   # floor is always 0.5, independent of class ratio
  secondary: best_f1             # max over a scan of all thresholds
  best_f1_trivial_floor: 0.8     # at a 1:2 class ratio, "predict all abnormal" already gives 0.8
  n_threshold_scan_points: 1000  # matches the linspace in baseline_runner.ipynb
  clip_score: mean_mse_over_frames   # per clip = mean reconstruction MSE over all its feature vectors
  reconstruction_mse_aggregation: median   # threshold shift ratio uses the median
"""
CONFIG_YAML = _CONFIG_YAML_RAW.lstrip("\n")     # strip the newline right after r""" so it matches the repo file byte for byte

CONFIG_SHA256 = hashlib.sha256(CONFIG_YAML.encode("utf-8")).hexdigest()

with open("/content/config.yaml", "w", encoding="utf-8") as f:
    f.write(CONFIG_YAML)

cfg = yaml.safe_load(CONFIG_YAML)


# --- The three functions below are identical to code/configs/config.py in the repo ---

def rng_for(seed, *parts):
    """Derive a stable Generator from (seed, parts...).

    Uses crc32 rather than the built-in hash(): the latter salts str randomly, so it changes across processes.
    """
    import zlib
    tag = "|".join(str(p) for p in parts).encode("utf-8")
    return np.random.default_rng([seed, zlib.crc32(tag)])


def nested_order(items, seed, *scope):
    """Ordering: sort by file name first (removes file-system enumeration order), then shuffle once with the derived RNG."""
    ordered = sorted(items)
    idx = rng_for(seed, *scope).permutation(len(ordered))
    return [ordered[i] for i in idx]


def self_check(cfg):
    src, tgt, grid, stages = cfg["source_domain"], cfg["target_domain"], cfg["grid"], cfg["stages"]

    n_eval_normal = src["eval_set"]["n_normal"]
    n_eval_abnormal = src["eval_set"]["n_abnormal"]
    for mid in src["machine_ids"]:
        counts = src["official_counts"][mid]
        expect = counts["normal"] - n_eval_normal
        got = src["train_pool_size"][mid]
        assert got == expect, f"{mid} train pool should be {expect}, yaml says {got}"
        assert counts["abnormal"] >= n_eval_abnormal, f"{mid} has fewer than {n_eval_abnormal} abnormal clips"

    merged, n_ids = src["merged_model"], len(src["machine_ids"])
    assert merged["eval_per_id"]["normal"] * n_ids == n_eval_normal
    assert merged["eval_per_id"]["abnormal"] * n_ids == n_eval_abnormal
    assert merged["train_per_id"] * n_ids == merged["train_total"]
    pools = list(src["train_pool_size"].values())
    assert min(pools) <= merged["train_total"] <= max(pools), "merged-model training size falls outside the single-ID range"
    assert merged["train_per_id"] <= min(pools)

    src_ratio = n_eval_abnormal / n_eval_normal
    tgt_ratio = tgt["n_abnormal_total"] / tgt["n_normal_total"]
    assert abs(src_ratio - tgt_ratio) < 1e-9, "source and target class ratios differ; best-F1 floors would differ"
    floor = 2 * src_ratio / (src_ratio + 1) / (1 + src_ratio / (src_ratio + 1))
    assert abs(floor - cfg["metrics"]["best_f1_trivial_floor"]) < 1e-9

    n_cells = len(tgt["conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_cells * tgt["n_runs"] == tgt["n_clips_total"] == 180
    n_normal_cells = len(tgt["normal_conditions"]) * len(tgt["voltages"]) * len(tgt["noise_levels"])
    assert n_normal_cells * tgt["n_runs"] == tgt["n_normal_total"]

    feat = cfg["features"]
    assert feat["n_mels"] * feat["frames"] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][0] == feat["input_dim"]
    assert cfg["model"]["decoder_dims"][-1] == feat["input_dim"]
    assert cfg["model"]["encoder_dims"][-1] == cfg["model"]["bottleneck_dim"]

    loro = cfg["loro"]
    assert loro["n_folds"] == tgt["n_runs"]
    assert loro["clips_per_fold"] == n_cells
    assert loro["normals_per_fold"] == n_normal_cells
    assert loro["normals_per_fold"] + loro["abnormals_per_fold"] == loro["clips_per_fold"]
    assert loro["max_train_normals"] == tgt["n_normal_total"] - loro["normals_per_fold"]
    assert max(grid["data_levels"]) <= loro["max_train_normals"]

    levels = grid["data_levels"]
    assert levels == sorted(set(levels))

    known = set(cfg["model"]["layer_index"]["encoder_linears"]) | set(cfg["model"]["layer_index"]["decoder_linears"])
    for name, scheme in grid["layer_schemes"].items():
        assert not set(scheme["trainable"]) - known, f"layer scheme {name} references a non-existent layer"
        assert scheme["trainable"]
    assert set(grid["layer_schemes"]["full"]["trainable"]) == known

    n_levels, n_schemes = len(levels), len(grid["layer_schemes"])
    small = stages["stage2_finetune"]["small_shift"]
    large = stages["stage2_finetune"]["large_shift"]
    assert small["n_configs"] == small["starts"] * small["targets_per_start"] * n_levels * n_schemes
    assert large["n_configs"] == large["starts"] * large["targets_per_start"] * n_levels * n_schemes
    assert small["n_runs"] == small["n_configs"] * small["folds"]
    assert large["n_runs"] == large["n_configs"] * large["folds"]
    assert stages["stage2_finetune"]["n_configs_total"] == small["n_configs"] + large["n_configs"]
    assert stages["stage2_finetune"]["n_runs_total"] == small["n_runs"] + large["n_runs"]

    s3 = stages["stage3_no_pretrain"]
    assert s3["n_configs"] == n_levels
    assert s3["n_runs"] == s3["n_configs"] * s3["folds"]
    assert cfg["total_runs"] == stages["stage2_finetune"]["n_runs_total"] + s3["n_runs"]

    s1 = stages["stage1_frozen_eval"]
    assert s1["small_shift_pairs"] == len(src["machine_ids"]) * (len(src["machine_ids"]) - 1)
    assert s1["large_shift_models"] == stages["stage0_pretrain"]["n_models"]
    assert s1["large_shift_folds"] == loro["n_folds"]
    assert small["starts"] == len(src["machine_ids"])
    assert large["starts"] == stages["stage0_pretrain"]["n_models"]


self_check(cfg)

SEED = cfg["seed"]
MACHINE_IDS = cfg["source_domain"]["machine_ids"]
FEAT = cfg["features"]
PRE = cfg["pretrain"]
P = cfg["paths"]["colab"]

print(f"Config loaded (proposal {cfg['proposal_version']}), sha256 = {CONFIG_SHA256[:16]}...")
print(f"Self-check passed ✅")
print(f"  seed        : {SEED}")
print(f"  machine IDs : {', '.join(MACHINE_IDS)}")
print(f"  eval set    : {cfg['source_domain']['eval_set']['n_normal']} normal + "
      f"{cfg['source_domain']['eval_set']['n_abnormal']} abnormal = 480 clips/model")
print(f"  features    : {FEAT['n_mels']} mel × {FEAT['frames']} frames = {FEAT['input_dim']} dims, "
      f"sr={FEAT['sr']}, n_fft={FEAT['n_fft']}, hop={FEAT['hop_length']}")
print(f"  pretrain    : lr={PRE['lr']}, batch={PRE['batch_size']}, epochs={PRE['epochs']}, "
      f"val_split={PRE['validation_split']}")

## 3 · Mount Drive and extract MIMII (idempotent)

Extracts to the **Colab local disk** `/content/data/mimii`, never back to Drive. The local disk is wiped when the runtime restarts;
re-running this cell then re-extracts automatically. If the data is already in place, extraction is skipped (idempotent).

This stage only uses `0_dB_fan.zip`; `raw_audio.zip` (target domain) is not needed in Stage 0 and is not extracted.

In [ ]:
from google.colab import drive

if not os.path.ismount("/content/drive"):
    drive.mount("/content/drive")
else:
    print("Drive already mounted")

MIMII_ZIP     = P["mimii_zip"]
MIMII_EXTRACT = P["mimii_extract_to"]
MIMII_ROOT    = P["mimii_root"]
OUTPUTS_ROOT  = P["outputs_root"]

# Output directories (on Drive, survive disconnects)
MODELS_DIR  = os.path.join(OUTPUTS_ROOT, cfg["outputs"]["pretrained_models_dir"])
LOGS_DIR    = os.path.join(OUTPUTS_ROOT, cfg["outputs"]["results_dir"])
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

if not os.path.exists(MIMII_ZIP):
    raise FileNotFoundError(
        f"❌ MIMII zip not found: {MIMII_ZIP}\n"
        f"   Actual contents of HVAC_fan/ on Drive:\n     "
        + "\n     ".join(sorted(os.listdir(os.path.dirname(MIMII_ZIP)))
                          if os.path.isdir(os.path.dirname(MIMII_ZIP)) else ["(directory does not exist)"])
        + "\n   → If the path differs, set cfg['paths']['colab']['mimii_zip'] and re-run this cell."
    )

zip_gb = os.path.getsize(MIMII_ZIP) / 1024**3
print(f"Zip    : {MIMII_ZIP}  ({zip_gb:.2f} GB)")

def _already_extracted():
    return all(os.path.isdir(os.path.join(MIMII_ROOT, mid, lab))
               for mid in MACHINE_IDS for lab in ("normal", "abnormal"))

if _already_extracted():
    print(f"✅ Data already in {MIMII_ROOT}, skipping extraction")
else:
    os.makedirs(MIMII_EXTRACT, exist_ok=True)
    print(f"Extracting to {MIMII_EXTRACT} ... (about 3-20 min depending on zip size; Drive reads are the bottleneck)")
    t0 = time.time()
    with zipfile.ZipFile(MIMII_ZIP) as zf:
        members = zf.infolist()
        for m in tqdm(members, desc="unzip", unit="file"):
            zf.extract(m, MIMII_EXTRACT)
    print(f"Extraction done in {time.time() - t0:.0f} s")

print(f"\nOutput directories (Drive):")
print(f"  weights : {MODELS_DIR}")
print(f"  logs    : {LOGS_DIR}")

## 4 · Directory layout and file-count checks

Two gates; if either fails, it **raises an explicit error and prints the actual structure**, never continuing silently:

1. **Layout**: automatically searches the extraction directory for a directory containing all of `id_00/id_02/id_04/id_06`, each with
   `normal/` and `abnormal/` subdirectories. If the path found differs from the config assumption, it prints the actual directory tree, raises,
   and shows the value to use instead.
2. **Counts**: the number of wav files per machine ID must exactly match `config.official_counts`
   (1011/407, 1016/359, 1033/348, 1015/361). On mismatch it prints an actual-vs-expected table and raises.

In [ ]:
def print_tree(base, max_depth=3, max_entries=12):
    base = os.path.abspath(base)
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        indent = "  " * depth
        name = os.path.basename(root) or root
        n_wav = sum(1 for f in files if f.lower().endswith(".wav"))
        extra = f"   [{n_wav} wav]" if n_wav else ""
        print(f"{indent}{name}/{extra}")
        dirs.sort()
        if len(dirs) > max_entries:
            print(f"{indent}  ... {len(dirs)} subdirectories, showing first {max_entries}")
            dirs[:] = dirs[:max_entries]


def detect_mimii_root(base):
    """Find the directory that contains all 4 machine IDs, each with normal/ and abnormal/ subdirectories."""
    for root, dirs, _ in os.walk(base):
        if all(os.path.isdir(os.path.join(root, mid, lab))
               for mid in MACHINE_IDS for lab in ("normal", "abnormal")):
            return root
    return None


# --- Gate 1: directory layout ----------------------------------------------
detected = detect_mimii_root(MIMII_EXTRACT)

if detected is None:
    print("❌ No directory matching the expected MIMII layout under the extraction directory.")
    print(f"   Expected: <some dir>/{{{','.join(MACHINE_IDS)}}}/{{normal,abnormal}}/*.wav")
    print(f"\n   Actual structure of {MIMII_EXTRACT}:")
    print_tree(MIMII_EXTRACT)
    raise RuntimeError(
        "MIMII directory layout does not match expectations -- this is a **data/path problem**, not a code problem."
        " Check the actual structure printed above and adjust the config accordingly."
    )

if os.path.realpath(detected) != os.path.realpath(MIMII_ROOT):
    print("❌ MIMII root directory does not match the config assumption.")
    print(f"   config assumes : {MIMII_ROOT}")
    print(f"   detected       : {detected}")
    print(f"\n   Actual structure of {MIMII_EXTRACT}:")
    print_tree(MIMII_EXTRACT)
    raise RuntimeError(
        f"Path mismatch -- this is a **path problem**. Quickest fix: insert this line after the previous cell\n"
        f'    MIMII_ROOT = "{detected}"\n'
        f"and continue running from this cell; or set paths.colab.mimii_root in the config to that value."
    )

print(f"✅ Layout check passed: {MIMII_ROOT}")

# --- Gate 2: file counts ---------------------------------------------------
FILES = {}
rows, mismatch = [], False
for mid in MACHINE_IDS:
    FILES[mid] = {}
    for lab in ("normal", "abnormal"):
        found = sorted(glob.glob(os.path.join(MIMII_ROOT, mid, lab, "*.wav")))
        FILES[mid][lab] = found
        exp = cfg["source_domain"]["official_counts"][mid][lab]
        ok = len(found) == exp
        mismatch |= not ok
        rows.append((mid, lab, exp, len(found), "✅" if ok else "❌"))

print(f"\n{'ID':<8}{'Label':<10}{'Exp.':>6}{'Found':>6}   ")
print("-" * 36)
for mid, lab, exp, got, flag in rows:
    print(f"{mid:<8}{lab:<10}{exp:>6}{got:>6}   {flag}")

if mismatch:
    raise RuntimeError(
        "File counts do not match config.official_counts -- this is a **data problem**"
        " (the zip is not the MIMII 0 dB fan set assumed by the proposal, or extraction is incomplete)."
        " Review the table above before deciding whether to change the config or replace the data."
    )

total = sum(len(v[l]) for v in FILES.values() for l in v)
print(f"\n✅ Count check passed, {total} wav files in total")

## 5 · Data split (evaluation set first, seed=42)

Strictly follows `proposal/proposal.md` §4.1:

- **Single-ID models**: first draw 160 normal + 320 abnormal clips as the evaluation set; train pool = remaining normal clips (851/856/873/855)
- **Merged model**: evaluation set = a **subset of each ID's already-fixed evaluation set** (40 normal + 80 abnormal per ID, not re-drawn independently);
  training set = 215 clips drawn from each ID's train pool, 860 in total
- Ordering rule matches `nested_order()` in `code/configs/config.py`: sort by file name first,
  then shuffle once with an RNG derived from `crc32(scope)`, and take the first k

The split is written to `stage0_splits_seed42.json` on Drive. **All later stages must reuse this list**,
so the evaluation set is always the same set of files across the whole project.

In [ ]:
def split_take(items, k, seed, *scope):
    """After ordering, take the first k as A and the rest as B. Both are returned sorted by file name for readability and reproducibility."""
    ordered = nested_order(items, seed, *scope)
    return sorted(ordered[:k]), sorted(ordered[k:])


N_EVAL_N = cfg["source_domain"]["eval_set"]["n_normal"]       # 160
N_EVAL_A = cfg["source_domain"]["eval_set"]["n_abnormal"]     # 320
MERGED   = cfg["source_domain"]["merged_model"]

splits = {}
for mid in MACHINE_IDS:
    eval_n, train_pool = split_take(FILES[mid]["normal"],   N_EVAL_N, SEED, "eval_normal", mid)
    eval_a, _          = split_take(FILES[mid]["abnormal"], N_EVAL_A, SEED, "eval_abnormal", mid)

    assert len(eval_n) == N_EVAL_N and len(eval_a) == N_EVAL_A
    assert len(train_pool) == cfg["source_domain"]["train_pool_size"][mid], (
        f"{mid} train pool {len(train_pool)} != config value "
        f"{cfg['source_domain']['train_pool_size'][mid]}")
    assert not (set(train_pool) & set(eval_n)), f"{mid} train pool overlaps the evaluation set"

    splits[mid] = {"train_pool": train_pool, "eval_normal": eval_n, "eval_abnormal": eval_a}

# Merged model: eval set must be nested in each ID's fixed eval set; training set is drawn from each ID's train pool
merged_train, merged_eval_n, merged_eval_a = [], [], []
for mid in MACHINE_IDS:
    s = splits[mid]
    merged_train  += nested_order(s["train_pool"],    SEED, "merged_train",   mid)[:MERGED["train_per_id"]]
    merged_eval_n += nested_order(s["eval_normal"],   SEED, "merged_eval_n",  mid)[:MERGED["eval_per_id"]["normal"]]
    merged_eval_a += nested_order(s["eval_abnormal"], SEED, "merged_eval_a",  mid)[:MERGED["eval_per_id"]["abnormal"]]

merged_train, merged_eval_n, merged_eval_a = sorted(merged_train), sorted(merged_eval_n), sorted(merged_eval_a)
assert len(merged_train) == MERGED["train_total"] == 860
assert len(merged_eval_n) == N_EVAL_N and len(merged_eval_a) == N_EVAL_A
for mid in MACHINE_IDS:                         # nesting: merged eval set ⊆ per-ID eval sets
    assert set(f for f in merged_eval_n if f"/{mid}/" in f) <= set(splits[mid]["eval_normal"])
    assert set(f for f in merged_eval_a if f"/{mid}/" in f) <= set(splits[mid]["eval_abnormal"])
assert not (set(merged_train) & (set(merged_eval_n) | set(merged_eval_a)))

splits["merged"] = {"train_pool": merged_train, "eval_normal": merged_eval_n, "eval_abnormal": merged_eval_a}

print(f"{'Model':<10}{'Train':>10}{'EvalNorm':>10}{'EvalAbn':>10}{'EvalTotal':>10}")
print("-" * 50)
for name in MACHINE_IDS + ["merged"]:
    s = splits[name]
    print(f"{name:<10}{len(s['train_pool']):>10}{len(s['eval_normal']):>10}"
          f"{len(s['eval_abnormal']):>10}{len(s['eval_normal']) + len(s['eval_abnormal']):>10}")

SPLITS_PATH = os.path.join(LOGS_DIR, f"stage0_splits_seed{SEED}.json")
with open(SPLITS_PATH, "w") as f:
    json.dump({
        "seed": SEED,
        "config_sha256": CONFIG_SHA256,
        "mimii_root": MIMII_ROOT,
        "splits": {k: {kk: [os.path.relpath(p, MIMII_ROOT) for p in vv] for kk, vv in v.items()}
                   for k, v in splits.items()},
    }, f, indent=2)
print(f"\n✅ Split list written to {SPLITS_PATH}")
print("   Later stages must reuse this list; do not re-split.")

## 6 · Feature extraction and `.npy` cache

Implementation requirement from Stage 2 of `proposal/proposal.md`: extract features once and reuse the cache; never re-run librosa for every training run.

**Extraction is identical to `baseline_runner.ipynb`**: `librosa.load(sr=None, mono=True)` →
64 mel (n_fft=1024, hop=512, power=2.0) → `20/power·log10(mel+eps)` → concatenate 5 adjacent frames into a 320-dim vector.
Each 10 s clip yields 309 vectors.

**Cache location: Colab local disk** `/content/data/features` (about 2.1 GB). Not written to Drive, because
a 2 GB round trip through Drive is slower than recomputing: recomputing all features takes only 3–8 minutes, whereas writing 2 GB to Drive usually takes well over ten minutes
and uses cloud quota. **After a runtime restart, re-run this cell to recompute** (the cell itself is idempotent and skips if the cache exists).

Each `(machine ID, label)` is stored as one `.npy` (all vectors stacked vertically) + one `.meta.json`
(file list and per-clip vector offsets); at evaluation time the offsets are used to slice the array back into clips.

In [ ]:
FEATURE_DIR = os.path.join(P["extract_root"], "features")
os.makedirs(FEATURE_DIR, exist_ok=True)


def file_to_vector_array(file_name, n_mels=None, frames=None, n_fft=None, hop_length=None, power=None):
    """Line-for-line identical to baseline_runner.ipynb."""
    n_mels     = FEAT["n_mels"]     if n_mels     is None else n_mels
    frames     = FEAT["frames"]     if frames     is None else frames
    n_fft      = FEAT["n_fft"]      if n_fft      is None else n_fft
    hop_length = FEAT["hop_length"] if hop_length is None else hop_length
    power      = FEAT["power"]      if power      is None else power

    y, sr = librosa.load(file_name, sr=None, mono=True)

    mel_spectrogram = librosa.feature.melspectrogram(
        y=y, sr=sr, n_fft=n_fft, hop_length=hop_length, n_mels=n_mels, power=power
    )
    log_mel_spectrogram = 20.0 / power * np.log10(mel_spectrogram + sys.float_info.epsilon)

    vectorarray_size = log_mel_spectrogram.shape[1] - frames + 1
    if vectorarray_size < 1:
        return np.empty((0, n_mels * frames), float)

    dims = n_mels * frames
    vectorarray = np.zeros((vectorarray_size, dims), float)
    for t in range(frames):
        vectorarray[:, n_mels * t: n_mels * (t + 1)] = log_mel_spectrogram[:, t: t + vectorarray_size].T
    return vectorarray


def build_cache(mid, label, files):
    """Extract all features for (mid, label) and save .npy + .meta.json. Skips if already present with a matching file list."""
    stem = os.path.join(FEATURE_DIR, f"mimii_{mid}_{label}")
    npy_path, meta_path = stem + ".npy", stem + ".meta.json"
    rel = [os.path.relpath(f, MIMII_ROOT) for f in files]

    if os.path.exists(npy_path) and os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        if meta["files"] == rel:
            return npy_path, meta

    chunks, offsets = [], [0]
    for f in tqdm(files, desc=f"{mid}/{label}", unit="clip", leave=False):
        v = file_to_vector_array(f).astype(np.float32)   # training uses float32 anyway, so store as float32
        chunks.append(v)
        offsets.append(offsets[-1] + len(v))

    X = np.concatenate(chunks, axis=0)
    del chunks
    np.save(npy_path, X)
    meta = {"files": rel, "offsets": offsets, "n_clips": len(files),
            "n_vectors": int(X.shape[0]), "dim": int(X.shape[1])}
    with open(meta_path, "w") as f:
        json.dump(meta, f)
    del X
    return npy_path, meta


# Files needed in Stage 0: all normal clips of each ID (train pool + eval normal) + 320 eval abnormal
t0 = time.time()
CACHE = {}
for mid in MACHINE_IDS:
    need_normal = sorted(set(splits[mid]["train_pool"]) | set(splits[mid]["eval_normal"]))
    need_abnormal = splits[mid]["eval_abnormal"]
    CACHE[mid] = {
        "normal":   build_cache(mid, "normal",   need_normal),
        "abnormal": build_cache(mid, "abnormal", need_abnormal),
    }
    n_v = CACHE[mid]["normal"][1]["n_vectors"] + CACHE[mid]["abnormal"][1]["n_vectors"]
    print(f"  {mid}: {len(need_normal)} normal + {len(need_abnormal)} abnormal clips → {n_v:,} vectors")

# Check vectors per clip are consistent (always 309 for 10 s audio)
per_clip = set()
for mid in MACHINE_IDS:
    for lab in ("normal", "abnormal"):
        off = CACHE[mid][lab][1]["offsets"]
        per_clip |= {off[i + 1] - off[i] for i in range(len(off) - 1)}
print(f"\nSet of vectors-per-clip values: {sorted(per_clip)}  (expected 309 for 10s@16kHz)")
if len(per_clip) > 1:
    print("⚠️ Clip lengths differ -- correctness unaffected (clips are sliced by offsets), but some recordings are not 10 s long")

cache_gb = sum(os.path.getsize(os.path.join(FEATURE_DIR, f))
               for f in os.listdir(FEATURE_DIR)) / 1024**3
print(f"✅ Feature cache ready: {FEATURE_DIR}  ({cache_gb:.2f} GB), took {time.time() - t0:.0f} s")
print("   Re-run this cell after a runtime restart (the local disk is wiped); it takes about as long.")

## 7 · Model, training and evaluation functions

Architecture and hyperparameters follow `baseline_runner.ipynb`: 320→64→64→**8**→64→64→320, ReLU (no activation on the output layer),
Adam (default lr=1e-3), MSELoss, batch 512, 50 epochs, 10% held out as validation for loss monitoring only.

**One implementation difference (algorithm unchanged)**: the training data is moved to the GPU once and batched manually with `randperm`
instead of a `DataLoader`. It is still "shuffled mini-batch Adam", but without the DataLoader's per-batch overhead,
which for a small ~50k-parameter model exceeds the compute itself. Seeds are fixed, so results are reproducible.

**Evaluation** (same as the baseline): each clip → per-frame MSE → mean over all frames of the clip = the clip's anomaly score;
AUC via `roc_auc_score`; best F1 = maximum over 1000 thresholds scanned on `[min, max]`.

In [ ]:
class MIMII_Baseline_AE(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 64), nn.ReLU(),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, 8),         nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(8, 64),         nn.ReLU(),
            nn.Linear(64, 64),        nn.ReLU(),
            nn.Linear(64, input_dim),          # official MIMII baseline: no activation on the output layer
        )

    def forward(self, x):
        return self.decoder(self.encoder(x))


def load_vectors(mid, label, files):
    """Fetch the vectors of the given files from the cache. Returns (X, clip_slices)."""
    npy_path, meta = CACHE[mid][label]
    X = np.load(npy_path, mmap_mode="r")
    index = {f: i for i, f in enumerate(meta["files"])}
    off = meta["offsets"]
    parts, slices, cur = [], [], 0
    for f in files:
        i = index[os.path.relpath(f, MIMII_ROOT)]
        seg = np.asarray(X[off[i]:off[i + 1]])
        parts.append(seg)
        slices.append((cur, cur + len(seg)))
        cur += len(seg)
    return np.concatenate(parts, axis=0), slices


def train_model(X_train, seed, tag):
    """X_train: (n_vectors, 320) float32. Returns the trained model."""
    torch.manual_seed(seed)
    np.random.seed(seed)

    model = MIMII_Baseline_AE(FEAT["input_dim"]).to(DEVICE)
    optimizer = optim.Adam(model.parameters(), lr=PRE["lr"])
    criterion = nn.MSELoss()

    X = torch.from_numpy(np.ascontiguousarray(X_train)).to(DEVICE)
    n = len(X)
    n_val = int(n * PRE["validation_split"])
    perm = torch.randperm(n, device=DEVICE)
    val_idx, train_idx = perm[:n_val], perm[n_val:]
    Xtr, Xva = X[train_idx], X[val_idx]
    del X, perm, val_idx, train_idx

    bs, epochs = PRE["batch_size"], PRE["epochs"]
    n_batches = (len(Xtr) + bs - 1) // bs
    print(f"    train vectors {len(Xtr):,} | val vectors {len(Xva):,} | {n_batches} batch/epoch "
          f"× {epochs} epoch = {n_batches * epochs:,} steps")

    t0 = time.time()
    for epoch in range(epochs):
        model.train()
        idx = torch.randperm(len(Xtr), device=DEVICE)
        total = 0.0
        for b in range(n_batches):
            xb = Xtr[idx[b * bs:(b + 1) * bs]]
            optimizer.zero_grad()
            loss = criterion(model(xb), xb)
            loss.backward()
            optimizer.step()
            total += loss.item()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                va = criterion(model(Xva), Xva).item() if len(Xva) else float("nan")
            print(f"    epoch {epoch + 1:>3}/{epochs} | train {total / n_batches:.6f} | "
                  f"val {va:.6f} | {time.time() - t0:5.1f}s")

    del Xtr, Xva
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()
    return model


@torch.no_grad()
def clip_scores(model, X, slices):
    """Per-clip anomaly score = mean reconstruction MSE over all frames of the clip (same as the baseline)."""
    model.eval()
    Xt = torch.from_numpy(np.ascontiguousarray(X)).to(DEVICE)
    out = torch.empty(len(slices), dtype=torch.float64)
    for k, (a, b) in enumerate(slices):
        seg = Xt[a:b]
        mse_per_frame = torch.mean((seg - model(seg)) ** 2, dim=1)
        out[k] = mse_per_frame.mean().double().cpu()
    del Xt
    return out.numpy()


def find_best_f1(y_true, y_scores, n_points):
    """Scan n_points thresholds and take the max F1 (same linspace logic as the baseline)."""
    best = {"f1": 0.0, "threshold": float("nan"), "precision": 0.0, "recall": 0.0, "accuracy": 0.0}
    for t in np.linspace(min(y_scores), max(y_scores), n_points):
        pred = (y_scores > t).astype(int)
        p, r, f1, _ = precision_recall_fscore_support(y_true, pred, average="binary", zero_division=0)
        if f1 > best["f1"]:
            best = {"f1": float(f1), "threshold": float(t), "precision": float(p),
                    "recall": float(r), "accuracy": float(accuracy_score(y_true, pred))}
    return best


def evaluate(model, name):
    """Evaluate on the model's own 480-clip source-domain evaluation set."""
    s = splits[name]
    if name == "merged":
        scores, labels, files = [], [], []
        for mid in MACHINE_IDS:
            for lab, y in (("eval_normal", 0), ("eval_abnormal", 1)):
                fs = [f for f in s[lab] if f"/{mid}/" in f]
                if not fs:
                    continue
                X, sl = load_vectors(mid, lab.replace("eval_", ""), fs)
                scores.append(clip_scores(model, X, sl))
                labels += [y] * len(fs)
                files += fs
        scores = np.concatenate(scores)
        labels = np.array(labels)
    else:
        Xn, sn = load_vectors(name, "normal",   s["eval_normal"])
        Xa, sa = load_vectors(name, "abnormal", s["eval_abnormal"])
        scores = np.concatenate([clip_scores(model, Xn, sn), clip_scores(model, Xa, sa)])
        labels = np.array([0] * len(s["eval_normal"]) + [1] * len(s["eval_abnormal"]))
        files = s["eval_normal"] + s["eval_abnormal"]

    auc = float(metrics.roc_auc_score(labels, scores))
    best = find_best_f1(labels, scores, cfg["metrics"]["n_threshold_scan_points"])
    return auc, best, scores, labels, files


print("✅ Model and training/evaluation functions defined")
print(f"   parameters: {sum(p.numel() for p in MIMII_Baseline_AE(FEAT['input_dim']).parameters()):,}")

## 8 · Train the 5 models in sequence

Order: `id_00 → id_02 → id_04 → id_06 → merged`.

**Each model's weights and results are written to Drive as soon as it finishes**, so a runtime disconnect midway does not lose everything;
re-running this cell skips models that are already done (`SKIP_IF_DONE = True`; set it to `False` to force retraining).

For each model it prints: training size → train/val loss every 5 epochs → a one-line summary (training clips, AUC, best F1).

In [ ]:
SKIP_IF_DONE = True     # skip if weights already exist; set to False to force retraining

ALL_MODELS = MACHINE_IDS + ["merged"]
results = {}

for k, name in enumerate(ALL_MODELS, 1):
    ckpt_path = os.path.join(MODELS_DIR, f"baseline_fan_{name}.pth")
    res_path  = os.path.join(LOGS_DIR,   f"stage0_result_{name}.json")

    print(f"\n{'=' * 64}")
    print(f"[{k}/{len(ALL_MODELS)}] {name}")
    print("=" * 64)

    if SKIP_IF_DONE and os.path.exists(ckpt_path) and os.path.exists(res_path):
        with open(res_path) as f:
            results[name] = json.load(f)
        r = results[name]
        print(f"  ⏭  already done, skipping (AUC={r['auc']:.4f}, best F1={r['best_f1']:.4f})")
        continue

    t0 = time.time()

    # --- Load training data ---
    if name == "merged":
        parts, n_clips = [], 0
        for mid in MACHINE_IDS:
            fs = [f for f in splits["merged"]["train_pool"] if f"/{mid}/" in f]
            X, _ = load_vectors(mid, "normal", fs)
            parts.append(X)
            n_clips += len(fs)
        X_train = np.concatenate(parts, axis=0)
        del parts
    else:
        fs = splits[name]["train_pool"]
        X_train, _ = load_vectors(name, "normal", fs)
        n_clips = len(fs)

    print(f"  training data: {n_clips} clips → {len(X_train):,} vectors of dim {X_train.shape[1]}")

    # --- Train ---
    model = train_model(X_train, SEED, name)
    train_sec = time.time() - t0
    del X_train

    # --- Evaluate (own 480-clip source-domain eval set) ---
    print("  evaluating ...")
    auc, best, scores, labels, files = evaluate(model, name)

    # --- Save to Drive immediately ---
    torch.save({
        "model_state_dict": model.state_dict(),
        "input_dim": FEAT["input_dim"],
        "model_name": name,
        "seed": SEED,
        "config_sha256": CONFIG_SHA256,
        "pretrain": PRE,
        "n_train_clips": n_clips,
        "auc": auc,
        "best_f1": best["f1"],
    }, ckpt_path)

    results[name] = {
        "model": name,
        "n_train_clips": n_clips,
        "n_eval_clips": len(labels),
        "auc": auc,
        "best_f1": best["f1"],
        "best_threshold": best["threshold"],
        "precision": best["precision"],
        "recall": best["recall"],
        "accuracy": best["accuracy"],
        "train_sec": round(train_sec, 1),
        "total_sec": round(time.time() - t0, 1),
    }
    with open(res_path, "w") as f:
        json.dump(results[name], f, indent=2)

    # Also save per-clip scores: Stage 0 already computes them, so Stage 1 need not recompute
    with open(os.path.join(LOGS_DIR, f"stage0_clip_scores_{name}.csv"), "w") as f:
        f.write("file,label,score\n")
        for fp, y, sc in zip(files, labels, scores):
            f.write(f"{os.path.relpath(fp, MIMII_ROOT)},{int(y)},{sc:.10e}\n")

    print(f"  ✅ {name}: trained on {n_clips} clips | AUC = {auc:.4f} | best F1 = {best['f1']:.4f} "
          f"| time {results[name]['total_sec']:.0f}s")
    print(f"     weights → {ckpt_path}")

    del model
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

print(f"\n{'=' * 64}\nAll {len(ALL_MODELS)} models done\n{'=' * 64}")

## 9 · Summary table (source-domain baseline)

These 5 rows are the **source-domain baseline** required by Stage 0 of `proposal/proposal.md`, i.e. the reference subtracted when Stage 2 computes "forgetting".
Written to `stage0_source_baseline.csv` on Drive.

For reference: the trivial floor of `best F1` is **0.800** (obtained by labelling every clip abnormal under the 1:2 class ratio),
so F1 reflects real discrimination only when clearly above 0.80; **AUC is the primary metric** (floor 0.5).

In [ ]:
import pandas as pd

df = pd.DataFrame([results[n] for n in ALL_MODELS])[
    ["model", "n_train_clips", "n_eval_clips", "auc", "best_f1",
     "precision", "recall", "accuracy", "total_sec"]
]

print("Source-domain baseline (Stage 0 output)")
print("=" * 88)
print(f"{'Model':<10}{'n_train':>9}{'n_eval':>9}{'AUC':>9}{'BestF1':>9}"
      f"{'Prec':>9}{'Recall':>9}{'Time(s)':>9}")
print("-" * 88)
for r in (results[n] for n in ALL_MODELS):
    print(f"{r['model']:<10}{r['n_train_clips']:>9}{r['n_eval_clips']:>9}"
          f"{r['auc']:>9.4f}{r['best_f1']:>9.4f}{r['precision']:>9.4f}"
          f"{r['recall']:>9.4f}{r['total_sec']:>9.0f}")
print("=" * 88)
print(f"{'Mean':<10}{'':>9}{'':>9}{df['auc'].mean():>9.4f}{df['best_f1'].mean():>9.4f}")
print(f"\nTrivial floor of best F1 = {cfg['metrics']['best_f1_trivial_floor']} (1:2 class ratio); "
      f"F1 must be clearly above this to reflect real discrimination")

CSV_PATH = os.path.join(LOGS_DIR, "stage0_source_baseline.csv")
df.to_csv(CSV_PATH, index=False)

print(f"\n✅ Stage 0 done. Outputs on Drive:")
print(f"   {MODELS_DIR}/")
for n in ALL_MODELS:
    print(f"     baseline_fan_{n}.pth")
print(f"   {LOGS_DIR}/")
print(f"     stage0_source_baseline.csv        ← source-domain baseline summary (this table)")
print(f"     stage0_splits_seed{SEED}.json          ← data split list, must be reused by later stages")
print(f"     stage0_result_<model>.json        × {len(ALL_MODELS)}")
print(f"     stage0_clip_scores_<model>.csv    × {len(ALL_MODELS)}  ← per-clip anomaly scores")
df